#init step

In [0]:
%python
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
%python
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

#Reading From Bronze Layer

In [0]:
%python
df = spark.table("workspace.bronze.crm_cust_info")

#Data Transformation

##Trimming

In [0]:
%python
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):  # Trim all string cols
        df = df.withColumn(field.name, trim(col(field.name)))

##Normalization

In [0]:
%python

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .otherwise("n/a")
    )
)

## Renaming The Columns

In [0]:
%python 
for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new) 

#Write Into Silver Layer

In [0]:
%python
( 
    df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("silver.crm_customers")
) 